In [1]:
import pandas as pd 
import numpy as np 

from scipy.stats import nbinom

1. To create a mapped unmapped table as we are missing 14 (DMSO) samples in the mapped unmapped table 

2. create a third replicate

In [2]:
path_to_count = "input/BIOS2931_gene_counts_hwt_wo_BOP.csv"
# path_to_mapped = "input/BIOS2931_mapped_unmapped_hwt.csv"
parameters = ""

In [3]:
count_df  = pd.read_csv(path_to_count, index_col=0)
# map_df  = pd.read_csv(path_to_mapped, index_col=0)


## 1.

In [4]:
## find overlapping cols for mapped table 

# overlap_col = [i for i in count_df.columns if i  in map_df.columns]
# print(len(overlap_col) ) 

# map_df_filtered= map_df[overlap_col]

In [5]:
# missing_samples = [i for i in count_df.columns if i not in map_df_filtered.columns]
# print(len(missing_samples))

# # get mapped read counts for the samples missing in map df 

# mapped_vals = count_df[missing_samples].sum(axis=0)


# unmapped_ratios = map_df_filtered.loc['unmapped'] / map_df_filtered.loc['TotalReads']
# mean_ratio = unmapped_ratios.mean()
# std_ratio = unmapped_ratios.std()

# new_rows = pd.DataFrame(index=['unmapped', 'mapped', 'TotalReads', 'PercentMapped'])
# for sample in missing_samples:
#     m = mapped_vals[sample]
#     # Sample a ratio from a normal distribution based on your existing data
#     ratio = np.clip(np.random.normal(mean_ratio, std_ratio), 0.01, 0.99)
#     u = int(m * ratio / (1 - ratio))
#     total = u + m
#     pct = round((m / total) * 100, 2)
#     new_rows[sample] = [u, m, total, pct]

# map_df_filtered = pd.concat([map_df_filtered, new_rows], axis=1)

In [6]:
# map_df_filtered[missing_samples].loc['PercentMapped'].mean()

In [7]:
# map_df_filtered.to_csv('BIOS2931_mapped_unmmaped_synthetic.csv')

In [8]:
4476722.00/5180867.00*100

86.40874201171349

The usual RNA-seq-style parameterization is:

mean: 
μ
μ

dispersion: 
ϕ
ϕ

variance: 
μ+ϕμ2
μ+ϕμ
2

So for each gene in a sample pair like MCF7_DMSO1-1 and MCF7_DMSO1-2, you can set:

μ=(x1+x2)/2
μ=(x
1
	​

+x
2
	​

)/2

ϕ=0.2
ϕ=0.2

Then draw one new count from a negative binomial with that mean and dispersion. This matches the common NB variance form used in count models. SciPy documents the NB parameterization and the relationship between mean/variance and n, p; NumPy also provides NB sampling.

The conversion to SciPy/NumPy-style parameters is:

n=1/ϕ
n=1/ϕ

p=n/(n+μ)
p=n/(n+μ)

because then the NB has mean 
μ=n(1−p)/p
μ=n(1−p)/p and variance 
μ+μ2/n=μ+ϕμ2
μ+μ
2
/n=μ+ϕμ
2

## 2.

In [9]:

def simulate_third_replicate(count_df, dispersion=0.2, random_state=42):
    """
    count_df: rows = genes, columns = samples
              expects paired columns like:
              MCF7_DMSO1-1, MCF7_DMSO1-2
    returns a copy with added -3 columns
    """
    rng = np.random.default_rng(random_state)
    out = count_df.copy()

    # find sample stems ending in -1 / -2
    stems = set()
    for col in count_df.columns:
        if col.endswith("-1"):
            if 'DMSO' in col :
                continue 
            stem = col[:-2]
            if f"{stem}-2" in count_df.columns:
                stems.add(stem)

    for stem in sorted(stems):
        c1 = f"{stem}-1"
        c2 = f"{stem}-2"
        c3 = f"{stem}-3"

        mu = (count_df[c1].astype(float) + count_df[c2].astype(float)) / 2.0
        phi = dispersion

        # NB params for scipy.stats.nbinom
        n = 1.0 / phi
        p = n / (n + mu)

        # draw one simulated count per gene
        sim = nbinom.rvs(n, p, random_state=rng)

        out[c3] = sim.astype(int)

    return out

In [10]:
# count_df_rep2 = count_df.loc[:, (count_df.columns.str.contains('-1') ) | (count_df.columns.str.contains('-2')) ]

In [11]:
count_df_sim = simulate_third_replicate(count_df, dispersion=0.2, random_state=123)

In [12]:
count_df_sim

,MCF7_AZD_0p1uM-2,MCF7_AZD_0p1uM-1,MCF7_AZD_1uM-2,MCF7_AZD_1uM-1,MCF7_AZD_10uM-2,MCF7_AZD_10uM-1,MCF7_DACTO_0p1uM-2,MCF7_DACTO_0p1uM-1,MCF7_DACTO_1uM-2,MCF7_DACTO_1uM-1,...,MDA_TOR1_1uM-3,MDA_TOR2_0p1uM-3,MDA_TOR2_10uM-3,MDA_TOR2_1uM-3,MDA_WYE_0p1uM-3,MDA_WYE_10uM-3,MDA_WYE_1uM-3,MDA_ZOTA_0p1uM-3,MDA_ZOTA_10uM-3,MDA_ZOTA_1uM-3
A2M_1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
AAMP_2,770,1195,1061,1129,706,597,592,479,559,332,...,462,211,374,115,14,36,41,203,254,230
AARS1_3,681,850,727,939,501,575,347,362,352,256,...,445,127,123,136,24,30,28,283,183,271
AATF_4,13,16,21,6,3,27,8,26,15,8,...,16,27,10,44,3,0,2,51,35,96
ABCB1_12,14,19,17,6,15,5,7,14,1,5,...,0,1,0,0,0,0,1,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
RPL7_92861,3,4,0,2,2,7,4,0,2,0,...,0,0,0,2,0,0,0,0,0,0
RPL4_93059,7,10,10,5,5,5,7,4,8,3,...,1,1,3,4,0,0,0,1,0,0
GNG10_93243,1,0,2,3,2,0,0,0,0,0,...,0,0,0,1,0,0,0,0,1,1
ACOT1_93339,0,3,0,1,0,0,0,0,2,0,...,0,0,0,0,0,0,0,0,0,0


## 3. create a mapped unmapped for the third replciate 

In [13]:
map_df = pd.read_csv('input/BIOS2931_mapped_unmmaped_synthetic_dmso.csv', index_col=0)

# rep3_missing_mapped = [i for i in count_df_sim.columns if i not in map_df.columns]
# print(len(rep3_missing_mapped))

In [14]:
def create_synthetic_mapping_stats(count_df, map_df, random_state=42, min_jitter=0.2):
    rng = np.random.default_rng(random_state)
    out = map_df.copy()

    mapped_from_counts = count_df.sum(axis=0)

    stems = set()
    for col in count_df.columns:
        if col.endswith("-3"):
            if "DMSO" in col:
                continue
            stem = col[:-2]
            if f"{stem}-1" in count_df.columns and f"{stem}-2" in count_df.columns:
                stems.add(stem)

    for stem in sorted(stems):
        s1, s2, s3 = f"{stem}-1", f"{stem}-2", f"{stem}-3"

        if s1 not in out.columns or s2 not in out.columns:
            print(f"Skipping {stem}: missing {s1} or {s2} in mapping table")
            continue

        mapped_3 = int(mapped_from_counts[s3])

        pm1 = float(out.loc["PercentMapped", s1])
        pm2 = float(out.loc["PercentMapped", s2])

        mean_pm = (pm1 + pm2) / 2.0
        half_range = max(abs(pm1 - pm2) / 2.0, min_jitter)

        pm3 = rng.uniform(mean_pm - half_range, mean_pm + half_range)
        pm3 = float(np.clip(pm3, 0.1, 99.9))

        totalreads_3 = int(round(mapped_3 / (pm3 / 100.0)))
        totalreads_3 = max(totalreads_3, mapped_3)
        unmapped_3 = totalreads_3 - mapped_3

        out[s3] = np.nan
        out.loc["mapped", s3] = mapped_3
        out.loc["unmapped", s3] = unmapped_3
        out.loc["TotalReads", s3] = totalreads_3
        out.loc["PercentMapped", s3] = pm3

    return out

In [15]:
# map_df = map_df.reset_index(names='metric')
map_df_sim = create_synthetic_mapping_stats(count_df_sim, map_df)

In [ ]:
# count_df_sim.to_csv('./input/BIOS2931_gene_counts_synthetic_HH.csv')
# map_df_sim.to_csv('./input/BIOS2931_mapped_unmapped_synthetic_HH.csv')

In [17]:
count_df_sim.shape

(22533, 258)

In [18]:
map_df_sim.shape

(4, 258)

In [22]:
map_df_sim.filter(regex='MCF7_AZD_0p1uM-')

,MCF7_AZD_0p1uM-2,MCF7_AZD_0p1uM-1,MCF7_AZD_0p1uM-3
unmapped,213506.00,279022.00,2.394450e+05
mapped,3330056.00,4134238.00,3.708258e+06
TotalReads,3543562.00,4413260.00,3.947703e+06
PercentMapped,93.97,93.68,9.393458e+01


In [25]:
3.708258/3.947703*100

93.93457410549881

In [24]:
count_df_sim.filter(regex='MCF7_AZD_0p1uM-').sum()

MCF7_AZD_0p1uM-2    3330056
MCF7_AZD_0p1uM-1    4134238
MCF7_AZD_0p1uM-3    3708258
dtype: int64